# Redis da zero: articoli ERP
Esegui le celle dall’alto verso il basso. Questo notebook usa il client sincrono: più semplice per studiare, senza FastAPI o AI.

Redis deve essere già in esecuzione su localhost:6379. Non si crea un database con CREATE DATABASE: usiamo il database logico 0 e una chiave dedicata `lezione:redis:articoli:v1`, che contiene un hash Redis (campi e valori).

Se hai già Docker, puoi avviare un server di esempio dal terminale:
```powershell
docker run -d --name redis-lezione -p 127.0.0.1:6379:6379 -v redis_lezione_data:/data redis:7 redis-server --appendonly yes
```
Il volume conserva i file oltre la vita del container; AOF abilita la persistenza. Il comando crea un container nuovo: non ripeterlo se esiste già. Per riavviarlo usa `docker start redis-lezione`.

Il notebook non cancella dati e non imposta scadenze. I salvataggi sopravvivono ai riavvii solo se il server è configurato per la persistenza.


In [ ]:
# Installa il client nell'ambiente Python del kernel Jupyter.
# Non installa il server Redis.
%pip install redis


In [ ]:
import json
import redis

# db=0 seleziona un database logico già disponibile, non ne crea uno.
r = redis.Redis(
    host='localhost', port=6379, db=0,
    decode_responses=True,  # riceve stringhe anziché bytes
    socket_connect_timeout=3, socket_timeout=3
)

try:
    print('Redis raggiungibile:', r.ping())
except redis.exceptions.ConnectionError as errore:
    raise RuntimeError('Avvia prima il server Redis su localhost:6379.') from errore


## 1. Crea il contenitore e scrivi i primi articoli
Redis crea la chiave alla prima scrittura. Usiamo un hash: una chiave contenitore, un campo per codice articolo, un valore JSON per i dettagli. HSETNX scrive soltanto se il campo non esiste: rieseguire la cella non sovrascrive gli articoli.


In [ ]:
CHIAVE = 'lezione:redis:articoli:v1'

articoli_iniziali = {
    'ART001': {'descrizione': 'Valvola', 'quantita': 12},
    'ART002': {'descrizione': 'Raccordo', 'quantita': 30},
}

for codice, dettagli in articoli_iniziali.items():
    inserito = r.hsetnx(CHIAVE, codice, json.dumps(dettagli, ensure_ascii=False))
    print(codice, 'inserito' if inserito else 'già presente: mantenuto')

print('Tipo struttura:', r.type(CHIAVE))
print('Numero articoli:', r.hlen(CHIAVE))


## 2. Leggi un articolo
HGET cerca un campo dentro l’hash. Se manca restituisce None. json.loads trasforma la stringa JSON in un dizionario Python.


In [ ]:
codice = 'ART001'
valore = r.hget(CHIAVE, codice)

if valore is None:
    print('Articolo non trovato:', codice)
else:
    articolo = json.loads(valore)
    print('Codice:', codice)
    print('Descrizione:', articolo['descrizione'])
    print('Quantità:', articolo['quantita'])


## 3. Cerca un dato e aggiungilo soltanto se manca
Alla prima esecuzione ART003 viene aggiunto; alla seconda viene letto senza sovrascriverlo. HSETNX è atomico: anche se un altro programma inserisce lo stesso codice dopo la nostra lettura, non sovrascriviamo il suo dato.


In [ ]:
codice_cercato = 'ART003'  # prova anche ART001
valore = r.hget(CHIAVE, codice_cercato)

if valore is not None:
    print('Trovato:', codice_cercato, json.loads(valore))
else:
    nuovo_articolo = {'descrizione': 'Guarnizione', 'quantita': 50}
    inserito = r.hsetnx(
        CHIAVE, codice_cercato,
        json.dumps(nuovo_articolo, ensure_ascii=False)
    )
    if inserito:
        print('Articolo assente: aggiunto senza modificare gli altri.')
    else:
        print('Articolo inserito nel frattempo da un altro processo.')
    print('Dato ora presente:', json.loads(r.hget(CHIAVE, codice_cercato)))


## 4. Visualizza tutto il contenitore della lezione
HGETALL recupera tutti i campi di questa chiave, non tutto il database. È appropriato per il piccolo esempio; per strutture grandi si preferisce una scansione incrementale.


In [ ]:
tutti = r.hgetall(CHIAVE)

for codice, valore_json in sorted(tutti.items()):
    dati = json.loads(valore_json)
    print(f"{codice} | {dati['descrizione']} | quantità: {dati['quantita']}")

print('Totale articoli:', len(tutti))
print('TTL del contenitore:', r.ttl(CHIAVE))  # -1 = esiste senza scadenza


## 5. Verifica che i dati restino dopo una nuova connessione
Chiudere il client non arresta Redis e non elimina i dati. Questa prova verifica soltanto la riconnessione, non la persistenza su disco.


In [ ]:
r.close()
r = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True,
                socket_connect_timeout=3, socket_timeout=3)
print('Articoli dopo riconnessione:', r.hlen(CHIAVE))
print('ART003:', r.hget(CHIAVE, 'ART003'))


## Esercizi e limiti
1. Riesegui la cella di ricerca: ART003 ora esiste.
2. Cerca ART001: il dato iniziale resta invariato.
3. Per cercare ART004 cambia anche descrizione e quantità del nuovo articolo.
4. Per provare la persistenza con il container dell’esempio: esegui `docker restart redis-lezione` nel terminale, attendi che sia pronto e riesegui le celle di connessione e visualizzazione.

Non usare FLUSHDB o FLUSHALL: cancellerebbero anche dati estranei alla lezione. Non esporre Redis su Internet per questo esercizio.

Un hash Redis è una struttura chiave/campo/valore, non una tabella relazionale: non stiamo usando SQL, join o vincoli ERP. Per un archivio ufficiale con audit e interrogazioni articolate resta valida la proposta SQL Server.

Fonti: https://redis.io/docs/latest/commands/hsetnx/ ; https://redis.io/docs/latest/commands/hget/ ; https://redis.io/docs/latest/commands/hgetall/ ; https://redis.io/docs/latest/operate/oss_and_stack/management/persistence/

Verifica effettuata sul file: struttura notebook e sintassi delle celle Python; nessuna connessione Redis eseguita nell’ambiente dell’utente.
